# 配置收益标签与执行标记对齐：固定配对训练

两组共享2018年起原生执行价格覆盖的训练样本、原分类标签与模型设置。仅比较配置H1收益定义；保留长期强参考，不能以弱短窗对照冒充整体改进。

In [1]:
from pathlib import Path
import os,json
import qlib
from src.workflow.execution_return_training import run
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/execution_return_training_20261015';plan=json.loads((card/'plan.json').read_text());qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
summary=run(root=root,card=card,plan=plan)
print({k:summary[k] for k in ['decision','new_fits','new_native_cases','gates']})


# 执行收益标签对齐：不晋升

完成真实训练和完整账户回测：旧日线收益标签与09:47至下一交易日09:47执行收益标签，各3个seed、300轮，共6次拟合、6个新增Qlib原生账户；复用12个参考账户。两组主训练样本均为2018-10-12至2022-12-21的1056条成熟样本，辅助训练20741条，截止2023-01-01。保留长期训练小模型作强参考。

回测为2023-01-03信号至2026-10-09退出，共944个区间；本金10万美元，IBKR Pro Tiered冻结费用口径，成本情景独立运行。

| 账户 | 0 bp净收益 | 2 bp净收益 | 5 bp净收益 | 2 bp监控最大回撤 |
|---|---:|---:|---:|---:|
| 同样本旧标签对照 | +376.04% | +370.06% | +428.74% | 26.35% |
| 执行收益标签候选 | +125.67% | +121.05% | +120.30% | 32.40% |
| 长历史冻结小模型 | +342.52% | +335.36% | +348.06% | 27.64% |
| SOXX | +387.75% | +387.73% | +387.70% | 43.39% |

候选在三个成本情景均低于配对对照、长期小模型及SOXX，监控回撤31.89%/32.40%/32.98%均超过30%；126日正超额窗口比例约25.64%至28.08%，未达70%。年际、滚动及bootstrap门均失败，不能证明稳定超额。旧标签对照5 bp高于2 bp来自独立账户路径的总体差异，未完成逐交易分解，不能据此声称高费用有益。

只改变配置损失中的两项H1资产收益，H5分类标签及成熟时间保持原义；初始化、训练样本、缩放器、网络、训练轮数、控制器、费用均配对。该变化同时涉及价格来源/调整口径和执行时间，不能归因于单纯17分钟。执行价格是历史研究代理，未认证为实际券商成交；监控回撤并非全分钟或逐笔最大回撤。旧历史窗口已反复用于研究，不是全新独立OOS。

决策为not_promoted_execution_return_alignment_failed。固定路线关闭，不更换时点、样本窗口、seed或epoch挽救。下一项需要先定位两套收益口径的差异是否集中于公司行动/异常价格日期，并验证标签构造，再另行冻结机制实验；不能从本轮推出所有执行标签均无效。生产模型未替换，稳定超额目标未完成。

数据及模型证据存于Qlib Recorder 162df4726af342e891fe3db54b13845f，NAS归档已冷恢复并逐对象核对哈希。主入口为本目录执行后的notebook、summary.json和backtest_benchmark_comparison.csv。
